# Sujet 3 · Ce que disent les discours : fréquences, mots-clés, concordance et entités nommées

**Cours « Intelligence artificielle »** · Master 2 (ANG, LGC, DID) · Université Yahia Farès, Médéa
**Mini-projet, sujet 3** (analyse de texte pour l'extraction d'informations dans des discours politiques) · **Chapitre 2** (lire et traiter des fichiers textes : fréquences, concordance) · **Durée :** environ 1 h 30

**Ce que vous allez faire**
- compter les mots de trois extraits de discours célèbres de Victor Hugo et de Jean Jaurès, puis repérer leurs mots les plus fréquents une fois écartés les mots vides ;
- comparer les extraits avec **TF-IDF** pour trouver les mots qui caractérisent chacun, puis revenir au texte grâce à une **concordance** (KWIC) ;
- repérer automatiquement les lieux, les personnes et les organisations (les **entités nommées**) avec spaCy, et juger ses erreurs.

**Mode d'emploi**
Exécutez les cellules de haut en bas : cliquez sur le bouton ▶ à gauche de chaque cellule, ou appuyez sur Maj+Entrée. Vous n'avez jamais à écrire de code. Modifiez seulement les champs des cellules intitulées « À MODIFIER ». Dans Colab, le menu « Exécution → Tout exécuter » lance tout le carnet d'un coup. Dans Jupyter, ces champs apparaissent comme de simples lignes de code : changez seulement la valeur placée après le signe =.

Avant chaque cellule de code, une question vous demande de **prédire** le résultat : notez votre réponse, exécutez, puis comparez avec la rubrique « Ce que vous devez voir ». Python écrit les nombres décimaux avec un point : 0.42 se lit 0,42.

> **Les textes.** Trois extraits courts (moins de 150 mots chacun) de discours du domaine public, en français original. Les coupures sont signalées par […], y compris les réactions de la salle notées dans les comptes rendus ; la ponctuation varie légèrement d'une édition à l'autre.
>
> **Connexion Internet.** Seules les cellules de l'étape 10 (entités nommées) téléchargent un modèle : elles fonctionnent dans Colab. Tout le reste fonctionne sans Internet.

## Étape 1 · Charger les outils

Cette cellule charge les outils du cours : `re` (les expressions régulières, pour découper et chercher), `Counter` (pour compter), pandas (tableaux), matplotlib (graphiques) et, dans scikit-learn, `TfidfVectorizer`, qui calcule les scores TF-IDF.

**Avant d'exécuter :** cette cellule va-t-elle afficher un tableau, un graphique, ou seulement un court message ?

In [ ]:
#@title Charger les outils (bibliothèques)
import re                                   # expressions régulières : découper, chercher
from collections import Counter             # compter les éléments d'une liste
import pandas as pd                         # tableaux de données
import matplotlib.pyplot as plt             # graphiques
from sklearn.feature_extraction.text import TfidfVectorizer   # scores TF-IDF
print("Outils chargés.")

**Ce que vous devez voir :** un seul message, `Outils chargés.`

**Question :** parmi les outils chargés, lequel sert à compter ? Dans quelle leçon du chapitre 2 l'avez-vous déjà rencontré ?

## Étape 2 · Le corpus : trois extraits de discours

Un **corpus** est un ensemble de textes réunis pour une question de recherche. Le nôtre compte trois extraits, chacun rangé dans une variable (`extrait_1`, `extrait_2`, `extrait_3`). Les triples guillemets `"""` permettent d'écrire un texte sur plusieurs lignes.

**Extrait 1 · Victor Hugo, « Détruire la misère »**, Assemblée nationale législative, Paris, 9 juillet 1849. Député, Hugo soutient la proposition d'Armand de Melun sur la prévoyance et l'assistance publiques.

**Avant d'exécuter :** la cellule compte les mots en coupant le texte aux espaces. Combien de mots, selon vous : environ 50, 120 ou 300 ?

In [ ]:
#@title Extrait 1 : Victor Hugo, « Détruire la misère » (1849)
extrait_1 = """Je ne suis pas, messieurs, de ceux qui croient qu'on peut supprimer la souffrance
en ce monde ; la souffrance est une loi divine ; mais je suis de ceux qui pensent et
qui affirment qu'on peut détruire la misère. […] Remarquez-le bien, messieurs, je ne
dis pas diminuer, amoindrir, limiter, circonscrire, je dis détruire. […] La misère est
une maladie du corps social comme la lèpre était une maladie du corps humain ; la
misère peut disparaître comme la lèpre a disparu. Détruire la misère ! oui, cela est
possible. Les législateurs et les gouvernants doivent y songer sans cesse ; car, en
pareille matière, tant que le possible n'est pas fait, le devoir n'est pas rempli."""
print("Extrait 1 :", len(extrait_1.split()), "mots (découpage aux espaces)")

**Ce que vous devez voir :** `Extrait 1 : 118 mots (découpage aux espaces)`.

**Question :** Hugo corrige son propre vocabulaire : « je ne dis pas diminuer, amoindrir, limiter, circonscrire, je dis détruire ». Quel effet cette correction produit-elle sur l'auditoire ?

**Extrait 2 · Jean Jaurès, « Discours à la jeunesse »**, lycée d'Albi, 30 juillet 1903. Jaurès, député du Tarn, préside la distribution des prix du lycée où il a été professeur de philosophie.

**Avant d'exécuter :** en lisant le texte de la cellule, repérez la formule que Jaurès répète en tête de phrase. Combien de fois apparaît-elle ?

In [ ]:
#@title Extrait 2 : Jean Jaurès, « Discours à la jeunesse » (1903)
extrait_2 = """Le courage, c'est d'être tout ensemble, et quel que soit le métier, un praticien et
un philosophe. Le courage, c'est de comprendre sa propre vie, de la préciser, de
l'approfondir, de l'établir et de la coordonner cependant à la vie générale. […] Le
courage, c'est d'aimer la vie et de regarder la mort d'un regard tranquille ; c'est
d'aller à l'idéal et de comprendre le réel ; c'est d'agir et de se donner aux grandes
causes sans savoir quelle récompense réserve à notre effort l'univers profond, ni s'il
lui réserve une récompense. Le courage, c'est de chercher la vérité et de la dire ;
c'est de ne pas subir la loi du mensonge triomphant qui passe, et de ne pas faire
écho, de notre âme, de notre bouche et de nos mains aux applaudissements imbéciles et
aux huées fanatiques."""
print("Extrait 2 :", len(extrait_2.split()), "mots (découpage aux espaces)")

**Ce que vous devez voir :** `Extrait 2 : 139 mots (découpage aux espaces)`. La formule « Le courage, c'est » ouvre quatre phrases de l'extrait : c'est une **anaphore**.

**Question :** parmi les définitions du courage que donne Jaurès, laquelle vous paraît la plus inattendue devant des lycéens ? Pourquoi ?

**Extrait 3 · Victor Hugo, discours d'ouverture du Congrès de la paix**, Paris, 21 août 1849. Hugo préside ce congrès international, qui réunit des partisans de la paix venus d'Europe et d'Amérique.

**Avant d'exécuter :** combien de noms de villes comptez-vous dans le texte de la cellule ?

In [ ]:
#@title Extrait 3 : Victor Hugo, discours au Congrès de la paix (1849)
extrait_3 = """Un jour viendra où la guerre paraîtra aussi absurde et sera aussi impossible entre
Paris et Londres, entre Pétersbourg et Berlin, entre Vienne et Turin, qu'elle serait
impossible et qu'elle paraîtrait absurde aujourd'hui entre Rouen et Amiens, entre
Boston et Philadelphie. […] Un jour viendra où il n'y aura plus d'autres champs de
bataille que les marchés s'ouvrant au commerce et les esprits s'ouvrant aux idées. Un
jour viendra où les boulets et les bombes seront remplacés par les votes, par le
suffrage universel des peuples, par le vénérable arbitrage d'un grand sénat souverain
qui sera à l'Europe ce que le parlement est à l'Angleterre, ce que la diète est à
l'Allemagne, ce que l'Assemblée législative est à la France !"""
print("Extrait 3 :", len(extrait_3.split()), "mots (découpage aux espaces)")

**Ce que vous devez voir :** `Extrait 3 : 121 mots (découpage aux espaces)`. On compte dix villes (Paris, Londres, Pétersbourg, Berlin, Vienne, Turin, Rouen, Amiens, Boston, Philadelphie) et quatre noms de continent ou de pays (l'Europe, l'Angleterre, l'Allemagne, la France). Gardez cette liste : à l'étape 10, spaCy devra la retrouver.

**Question :** pourquoi Hugo associe-t-il des villes deux par deux (Paris et Londres, Rouen et Amiens…) ? Que veut-il faire paraître « absurde » ?

## Étape 3 · Rassembler le corpus et sa fiche descriptive

Le mini-projet demande de noter, pour chaque discours, l'orateur, la date, le lieu et la langue, et de préciser s'il s'agit de l'original ou d'une traduction. Cette cellule range ces **métadonnées** dans un tableau pandas, avec une ligne par extrait.

**Avant d'exécuter :** combien de lignes et combien de colonnes aura ce tableau ?

In [ ]:
#@title La fiche descriptive du corpus (métadonnées)
fiches = pd.DataFrame({
    "orateur": ["Victor Hugo", "Jean Jaurès", "Victor Hugo"],
    "date": ["9 juillet 1849", "30 juillet 1903", "21 août 1849"],
    "lieu": ["Paris, Assemblée législative", "Albi, lycée", "Paris, Congrès de la paix"],
    "langue": ["français (original)", "français (original)", "français (original)"],
}, index=["Hugo 1849, la misère", "Jaurès 1903, le courage", "Hugo 1849, la paix"])
fiches

**Ce que vous devez voir :** un tableau de 3 lignes et 4 colonnes (orateur, date, lieu, langue). Les titres courts des lignes (« Hugo 1849, la misère »…) serviront d'étiquettes dans toute la suite.

**Question :** pourquoi est-il important, dans une étude de discours, de savoir si l'on travaille sur l'original ou sur une traduction ? Pensez au comptage des mots.

La cellule suivante construit le **corpus** : un dictionnaire qui associe à chaque titre le texte de l'extrait. Elle vous permet aussi, si vous le souhaitez, d'ajouter un quatrième extrait (un court passage d'un discours du domaine public, ou un paragraphe écrit par vous). **Pour l'instant, laissez le champ `mon_extrait` vide.**

**Avant d'exécuter :** si le champ reste vide, combien d'extraits le programme annoncera-t-il ?

In [ ]:
#@title À MODIFIER (facultatif) : ajouter votre propre extrait au corpus
mon_titre = "Mon extrait" #@param {type:"string"}
mon_extrait = "" #@param {type:"string"}
corpus = dict(zip(fiches.index, [extrait_1, extrait_2, extrait_3]))   # titre -> texte
if mon_extrait.strip() != "":            # le champ n'est pas vide
    corpus[mon_titre] = mon_extrait       # on ajoute un 4e extrait
print(len(corpus), "extraits dans le corpus :")
for titre in corpus:
    print("  -", titre)

**Ce que vous devez voir :**
```
3 extraits dans le corpus :
  - Hugo 1849, la misère
  - Jaurès 1903, le courage
  - Hugo 1849, la paix
```
**Question :** trois extraits d'une centaine de mots forment un tout petit corpus. Combien de discours faudrait-il, selon vous, pour répondre à la question de recherche du mini-projet (les thèmes du débat général de l'ONU) ?

## Étape 4 · Découper en mots et compter

Couper aux espaces est approximatif : « ; », « ! » ou « […] » deviennent des « mots ». La fonction `mots_de` utilise plutôt l'expression régulière `\w+` (« une suite de lettres ou de chiffres »), comme au chapitre 2, après avoir tout mis en minuscules. Le tableau compte ensuite, pour chaque extrait :
- les **mots** (les occurrences, *tokens*) : chaque apparition compte ;
- les **formes différentes** (*types*) : « misère » écrit quatre fois ne compte qu'une fois ;
- le rapport **formes / mots**, une mesure simple de la variété du vocabulaire.

La cellule affiche aussi les 12 premiers mots de Jaurès tels que Python les découpe.

**Avant d'exécuter :** Jaurès comptait 139 mots avec le découpage aux espaces. Le découpage `\w+` en trouvera-t-il plus ou moins ? Regardez « c'est » et « d'être ».

In [ ]:
#@title Découper chaque extrait en mots, puis compter mots et formes
def mots_de(texte):
    """Liste des mots d'un texte, en minuscules (suites de lettres ou de chiffres)."""
    return re.findall(r"\w+", texte.lower())

print(mots_de(extrait_2)[:12])            # les 12 premiers mots de Jaurès
tailles = pd.DataFrame(index=list(corpus))
tailles["mots"] = [len(mots_de(texte)) for texte in corpus.values()]
tailles["formes différentes"] = [len(set(mots_de(texte))) for texte in corpus.values()]
tailles["formes / mots"] = (tailles["formes différentes"] / tailles["mots"]).round(2)
tailles

**Ce que vous devez voir :** la liste `['le', 'courage', 'c', 'est', 'd', 'être', 'tout', 'ensemble', 'et', 'quel', 'que', 'soit']`, puis ce tableau :

| | mots | formes différentes | formes / mots |
|---|---|---|---|
| Hugo 1849, la misère | 116 | 67 | 0.58 |
| Jaurès 1903, le courage | 152 | 80 | 0.53 |
| Hugo 1849, la paix | 131 | 78 | 0.60 |

Jaurès passe de 139 à 152 mots : l'apostrophe coupe « c'est » en `c` + `est` et « d'être » en `d` + `être`, une difficulté classique du français (l'élision). Chez Hugo (misère), c'est l'inverse : 116 au lieu de 118, car « ; », « ! » et « […] » ne sont plus comptés, même si « qu'on » et « n'est » comptent désormais pour deux.

**Question :** Jaurès a le rapport formes / mots le plus bas (0.53). Quel procédé, repéré à l'étape 2, l'explique en partie ?

## Étape 5 · Les mots les plus fréquents de tout le corpus

La cellule réunit les mots des trois extraits dans une seule liste, puis `Counter` compte chaque forme et `most_common(10)` donne les dix plus fréquentes, sous forme de paires (mot, nombre).

**Avant d'exécuter :** notez les trois mots qui, selon vous, arriveront en tête. Des mots comme « misère », « courage » ou « guerre » ont-ils une chance ?

In [ ]:
#@title Les 10 mots les plus fréquents de tout le corpus
tous_les_mots = []
for texte in corpus.values():
    tous_les_mots = tous_les_mots + mots_de(texte)    # on ajoute les mots de chaque extrait
compte = Counter(tous_les_mots)
print(len(tous_les_mots), "mots au total,", len(compte), "formes différentes")
print(compte.most_common(10))

**Ce que vous devez voir :**
```
399 mots au total, 193 formes différentes
[('et', 21), ('la', 19), ('de', 18), ('est', 15), ('le', 12), ('l', 8), ('les', 7), ('c', 7), ('d', 7), ('un', 7)]
```
Les dix premières places reviennent à des **mots vides** (*stop words*) : articles, prépositions, conjonctions, le verbe « être », et même des restes d'élision (`l`, `c`, `d`). Ils sont présents dans tous les textes et ne disent rien du sujet.

**Question :** ces mots sont-ils vraiment inutiles pour tout le monde ? Pensez à un didacticien qui étudie l'emploi des articles, ou à un stylisticien qui étudie le « je » de l'orateur.

## Étape 6 · Écarter les mots vides

Une **liste de mots vides** est une liste de mots outils à ignorer. La nôtre, écrite directement dans la cellule, en compte 91. Vous pouvez en ajouter dans le champ `mots_vides_en_plus`, séparés par des virgules. **Pour l'instant, laissez-le vide.**

**Avant d'exécuter :** que va afficher la cellule, si le champ reste vide ?

In [ ]:
#@title À MODIFIER : la liste des mots vides (mots outils à ignorer)
mots_vides_en_plus = "" #@param {type:"string"}
mots_vides = set("""le la les l un une des de du d au aux à et ou mais ni car que qu qui quoi
dont où ce c cet cette ces ceux celui celle se s sa son ses leur leurs notre nos
votre vos mon ma mes il ils elle elles on je j me m moi tu te t lui nous vous y en
ne n pas plus suis est sont était être a ont avoir sera dans par pour sur sans avec
entre comme aussi tout tous cela là si""".split())
for mot in mots_vides_en_plus.replace(",", " ").lower().split():
    mots_vides.add(mot)                   # on ajoute les mots du champ
print(len(mots_vides), "mots vides dans la liste.")

**Ce que vous devez voir :** `91 mots vides dans la liste.` Si vous ajoutez des mots dans le champ, le nombre augmente d'autant.

**Question :** la liste contient `ne` et `pas`. Que perd-on en écartant la négation, dans un discours comme celui de Hugo (« je ne dis pas diminuer ») ?

La cellule suivante garde, dans chaque extrait, les **mots pleins** (les mots qui ne sont pas dans la liste des mots vides), puis affiche les huit plus fréquents de chaque extrait, avec leur nombre d'occurrences entre parenthèses. Les lignes du tableau sont numérotées à partir de 0, comme toujours en Python.

**Avant d'exécuter :** quel mot arrivera en tête chez Jaurès ? Et dans le discours de Hugo sur la paix ?

In [ ]:
#@title Les 8 mots pleins les plus fréquents de chaque extrait
classement = {}
for titre, texte in corpus.items():
    pleins = [m for m in mots_de(texte) if m not in mots_vides]    # on retire les mots vides
    premiers = Counter(pleins).most_common(8)
    classement[titre] = pd.Series([f"{mot} ({n})" for mot, n in premiers])
pd.DataFrame(classement)

**Ce que vous devez voir :**

| | Hugo 1849, la misère | Jaurès 1903, le courage | Hugo 1849, la paix |
|---|---|---|---|
| 0 | misère (4) | courage (4) | jour (3) |
| 1 | peut (3) | vie (3) | viendra (3) |
| 2 | détruire (3) | comprendre (2) | absurde (2) |
| 3 | messieurs (2) | récompense (2) | impossible (2) |
| 4 | souffrance (2) | réserve (2) | ouvrant (2) |
| 5 | dis (2) | ensemble (1) | guerre (1) |
| 6 | maladie (2) | quel (1) | paraîtra (1) |
| 7 | corps (2) | soit (1) | paris (1) |

Les mots pleins font apparaître le thème (« misère », « courage ») et la rhétorique : les anaphores « Le courage, c'est » et « Un jour viendra » placent `courage`, `jour` et `viendra` en tête. À égalité, `Counter` garde l'ordre d'apparition dans le texte.

**Question :** chez Hugo (misère), `peut` (3) arrive avant `souffrance`. Relisez l'extrait : de quoi Hugo veut-il convaincre l'Assemblée avec ce verbe et avec le mot « possible » ?

## Étape 7 · Un graphique en barres

Un diagramme en barres rend le classement lisible d'un coup d'œil. Choisissez l'extrait dans la liste déroulante et le nombre de mots avec le curseur : la cellule compte les mots pleins de cet extrait et dessine une barre par mot, la plus longue en haut, avec son nombre au bout.

**Avant d'exécuter :** pour l'extrait choisi par défaut (Hugo, la misère), combien de barres auront exactement la même longueur que celle de « souffrance » ?

In [ ]:
#@title À MODIFIER : l'extrait et le nombre de mots du graphique
extrait_choisi = "Hugo 1849, la misère" #@param ["Hugo 1849, la misère", "Jaurès 1903, le courage", "Hugo 1849, la paix"] {allow-input: true}
nombre_de_mots = 10 #@param {type:"slider", min:5, max:20, step:1}
pleins = [m for m in mots_de(corpus[extrait_choisi]) if m not in mots_vides]
premiers = Counter(pleins).most_common(nombre_de_mots)      # paires (mot, nombre)
mots = [mot for mot, n in premiers]
nombres = [n for mot, n in premiers]
fig, ax = plt.subplots(figsize=(7, 1 + 0.35 * len(mots)))
barres = ax.barh(mots, nombres, color="#2a78d6", height=0.6)
ax.bar_label(barres, padding=3)            # le nombre au bout de chaque barre
ax.invert_yaxis()                          # le mot le plus fréquent en haut
ax.set_xticks(range(max(nombres) + 1))     # graduations entières : 0, 1, 2...
ax.set_xlabel("nombre d'occurrences")
ax.set_title(f"{extrait_choisi} : les mots pleins les plus fréquents")
plt.show()

**Ce que vous devez voir :** dix barres bleues. `misère` en haut (4), puis `peut` et `détruire` (3), puis sept mots à 2 : `messieurs`, `souffrance`, `dis`, `maladie`, `corps`, `lèpre`, `possible`. Six autres barres ont donc la même longueur que celle de « souffrance ».

**Question :** avec si peu de niveaux (4, 3, 2), le graphique distingue-t-il vraiment les mots ? Choisissez « Jaurès 1903, le courage » et 15 mots : que remarquez-vous à partir du sixième mot ?

## Étape 8 · Comparer les extraits avec TF-IDF

Les fréquences décrivent chaque extrait isolément. **TF-IDF** compare : un mot est **caractéristique** d'un extrait s'il y est fréquent **et** rare dans les autres.
- **TF** (*term frequency*) : le nombre d'occurrences du mot dans l'extrait.
- **IDF** (*inverse document frequency*) : la « rareté » du mot dans le corpus. Avec la formule de scikit-learn, IDF = ln((1 + N) / (1 + n)) + 1, où N est le nombre d'extraits et n le nombre d'extraits qui contiennent le mot. Pour N = 3 : un mot présent dans les 3 extraits vaut 1.00, dans 2 extraits 1.29, dans un seul 1.69.
- Le score TF-IDF est le produit TF × IDF ; scikit-learn le ramène ensuite à une échelle commune, pour que les extraits longs et courts restent comparables.

La première cellule calcule l'IDF de quelques mots, en gardant ici tous les mots, mots vides compris.

**Avant d'exécuter :** classez `et`, `loi` et `misère` du plus « rare » au plus « commun » dans le corpus.

In [ ]:
#@title À MODIFIER : la « rareté » (IDF) de quelques mots
mots_a_examiner = "et, loi, misère" #@param {type:"string"}
tous = TfidfVectorizer(token_pattern=r"\w+").fit(corpus.values())   # ici, on garde tous les mots
idf = pd.Series(tous.idf_, index=tous.get_feature_names_out())
for mot in mots_a_examiner.replace(",", " ").lower().split():
    presents = sum(1 for texte in corpus.values() if mot in mots_de(texte))   # nb d'extraits
    if mot in idf.index:
        print(f"{mot:<10} dans {presents} extrait(s) sur {len(corpus)}   IDF = {idf[mot]:.2f}")
    else:
        print(f"{mot:<10} absent du corpus")

**Ce que vous devez voir :**
```
et         dans 3 extrait(s) sur 3   IDF = 1.00
loi        dans 2 extrait(s) sur 3   IDF = 1.29
misère     dans 1 extrait(s) sur 3   IDF = 1.69
```
`loi` est le seul mot plein que partagent deux extraits (Hugo : « la souffrance est une loi divine » ; Jaurès : « la loi du mensonge triomphant »). Nos trois extraits n'ont presque aucun vocabulaire commun.

**Question :** sur un corpus de 50 discours, un mot présent dans un seul discours aurait un IDF de ln(51 / 2) + 1, soit environ 4.2. Pourquoi TF-IDF devient-il plus utile quand le corpus grandit ?

La cellule suivante calcule les scores TF-IDF de tous les mots de chaque extrait, puis affiche les huit mots les plus caractéristiques de chacun, avec leur score entre parenthèses. La case `retirer_les_mots_vides` (cochée) applique notre liste de mots vides.

**Avant d'exécuter :** le classement TF-IDF de Hugo (misère) sera-t-il très différent du classement par fréquence de l'étape 6 ? Aidez-vous du résultat précédent.

In [ ]:
#@title Les 8 mots les plus caractéristiques de chaque extrait (TF-IDF)
retirer_les_mots_vides = True #@param {type:"boolean"}
if retirer_les_mots_vides:
    vectoriseur = TfidfVectorizer(token_pattern=r"\w+", stop_words=sorted(mots_vides))
else:
    vectoriseur = TfidfVectorizer(token_pattern=r"\w+")
matrice = vectoriseur.fit_transform(corpus.values())     # une ligne par extrait, une colonne par mot
scores = pd.DataFrame(matrice.toarray(), index=list(corpus), columns=vectoriseur.get_feature_names_out())
caracteristiques = {}
for titre in corpus:
    meilleurs = scores.loc[titre].sort_values(ascending=False, kind="stable").head(8)
    caracteristiques[titre] = [f"{mot} ({s:.2f})" for mot, s in meilleurs.items()]
pd.DataFrame(caracteristiques)

**Ce que vous devez voir :**

| | Hugo 1849, la misère | Jaurès 1903, le courage | Hugo 1849, la paix |
|---|---|---|---|
| 0 | misère (0.42) | courage (0.44) | jour (0.35) |
| 1 | détruire (0.32) | vie (0.33) | viendra (0.35) |
| 2 | peut (0.32) | comprendre (0.22) | absurde (0.23) |
| 3 | corps (0.21) | récompense (0.22) | impossible (0.23) |
| 4 | dis (0.21) | réserve (0.22) | ouvrant (0.23) |
| 5 | lèpre (0.21) | agir (0.11) | allemagne (0.12) |
| 6 | maladie (0.21) | aimer (0.11) | amiens (0.12) |
| 7 | messieurs (0.21) | aller (0.11) | angleterre (0.12) |

Le classement ressemble beaucoup à celui des fréquences : comme nos extraits ne partagent presque aucun mot plein, l'IDF favorise tous les mots de la même façon. À égalité de score, les mots sont rangés par ordre alphabétique. Décochez la case et relancez : sans liste de mots vides, `la`, `de`, `et` reviennent en tête, parce qu'avec trois extraits l'IDF ne peut diviser l'importance d'un mot commun que par 1.69 au plus.

**Question :** d'après ce tableau, quels trois mots résumeraient le mieux chaque extrait ? Ces mots sont-ils des thèmes, des procédés rhétoriques, ou les deux ?

## Étape 9 · Revenir au texte : la concordance (KWIC)

Un mot compté hors de son contexte peut tromper. La **concordance** KWIC (*keyword in context*, « mot-clé en contexte ») affiche chaque occurrence d'un mot au milieu de son contexte gauche et droit, toutes alignées sur le mot recherché : c'est la fonction `kwic` du chapitre 2, adaptée à la prose. Nous lui ajoutons une commodité, comme dans les logiciels AntConc ou TXM : l'étoile `*` remplace n'importe quel début ou fin de mot (`peupl*` trouve peuple, peuples, peuplé).

Cette cellule **définit** la fonction, puis l'essaie sur l'extrait 1 avec le mot « misère ».

**Avant d'exécuter :** combien de lignes la concordance de « misère » affichera-t-elle ?

In [ ]:
#@title Définir la fonction de concordance (KWIC) et l'essayer
def kwic(texte, mot, largeur=35):
    """Chaque occurrence de mot, avec son contexte gauche et droit (* = n'importe quelles lettres)."""
    plat = " ".join(texte.split())                          # tout l'extrait sur une seule ligne
    motif = r"\b" + mot.replace("*", r"\w*") + r"\b"        # peupl* devient peupl\w*
    lignes = []
    for m in re.finditer(motif, plat, flags=re.IGNORECASE):
        gauche = plat[max(0, m.start() - largeur):m.start()]
        droite = plat[m.end():m.end() + largeur]
        lignes.append(f"{gauche:>{largeur}}[{m.group()}]{droite}")
    return lignes

for ligne in kwic(extrait_1, "misère"):
    print(ligne)

**Ce que vous devez voir :** quatre lignes, alignées sur le mot entre crochets :
```
i affirment qu'on peut détruire la [misère]. […] Remarquez-le bien, messieurs,
conscrire, je dis détruire. […] La [misère] est une maladie du corps social co
t une maladie du corps humain ; la [misère] peut disparaître comme la lèpre a
me la lèpre a disparu. Détruire la [misère] ! oui, cela est possible. Les légi
```
La lecture verticale montre ce que la fréquence (4) ne disait pas : « misère » est deux fois complément du verbe « détruire » et deux fois sujet (« La misère est une maladie », « la misère peut disparaître »).

**Question :** entre la première et la dernière ligne, « détruire la misère » passe d'une affirmation (« on peut ») à une exclamation. Quel effet cette reprise produit-elle ?

Maintenant, cherchez dans **tout le corpus**. Tapez un mot (ou un début de mot suivi de `*`) dans le champ `mot_cherche` ; le curseur `largeur` règle le nombre de caractères de contexte de chaque côté.

**Avant d'exécuter :** le mot `loi` apparaît dans deux extraits. Le sens est-il le même chez Hugo et chez Jaurès ?

In [ ]:
#@title À MODIFIER : le mot à chercher dans tout le corpus
mot_cherche = "loi" #@param {type:"string"}
largeur = 35 #@param {type:"slider", min:20, max:60, step:5}
for titre, texte in corpus.items():
    concordance = kwic(texte, mot_cherche, largeur)
    print(f"--- {titre} : {len(concordance)} occurrence(s)")
    for ligne in concordance:
        print(ligne)

**Ce que vous devez voir :**
```
--- Hugo 1849, la misère : 1 occurrence(s)
n ce monde ; la souffrance est une [loi] divine ; mais je suis de ceux qui
--- Jaurès 1903, le courage : 1 occurrence(s)
la dire ; c'est de ne pas subir la [loi] du mensonge triomphant qui passe,
--- Hugo 1849, la paix : 0 occurrence(s)
```
Le sens n'est pas le même : chez Hugo, la loi est « divine » et on ne peut pas la supprimer ; chez Jaurès, c'est la loi du mensonge, qu'il ne faut pas « subir ».

**Question :** un même mot, deux sens. Que risque-t-on si l'on se contente de compter « loi » dans un corpus de discours, sans lire la concordance ?

## Étape 10 · Les entités nommées avec spaCy (en ligne)

Une **entité nommée** désigne un être ou un objet précis : une personne, un lieu, une organisation. La reconnaissance d'entités nommées est une tâche d'**apprentissage automatique** : le modèle `fr_core_news_sm` de spaCy a appris à les repérer sur des milliers de phrases annotées. Il reconnaît quatre types :

| Étiquette | Signification | Exemple |
|---|---|---|
| `PER` | personne | Karim Benali |
| `LOC` | lieu (ville, pays, région) | Paris, l'Angleterre |
| `ORG` | organisation | l'UNESCO |
| `MISC` | divers (événements, œuvres, nationalités…) | |

Il ne repère pas les dates (voir la leçon du mini-projet).

> **Nécessite une connexion Internet ; fonctionne dans Colab.** Les quatre cellules de cette étape téléchargent spaCy et son modèle français (une quinzaine de Mo). Hors ligne, lisez simplement les résultats attendus. Aucune cellule placée après elles n'en dépend.

La première cellule installe spaCy (déjà présent dans Colab) et télécharge le modèle.

**Avant d'exécuter :** cette cellule va-t-elle analyser nos extraits ?

In [ ]:
#@title Installer spaCy et télécharger le modèle français (connexion Internet)
%pip install -q spacy
!python -m spacy download fr_core_news_sm

**Ce que vous devez voir :** des lignes de téléchargement, puis `Download and installation successful` et `You can now load the package via spacy.load('fr_core_news_sm')`. Rien n'est encore analysé. Si la cellule suivante signale que le modèle est introuvable, choisissez « Exécution → Redémarrer la session », puis relancez tout le carnet.

**Question :** pourquoi un modèle d'apprentissage automatique doit-il être téléchargé, alors que notre liste de mots vides tenait dans la cellule ?

La cellule suivante charge le modèle, analyse chaque extrait et range chaque entité trouvée dans un tableau : l'extrait, le texte de l'entité et son type.

**Avant d'exécuter :** reprenez votre liste de l'étape 2 (dix villes et quatre noms de pays ou de continent chez Hugo, la paix). spaCy les trouvera-t-il toutes ? Trouvera-t-il des entités chez Jaurès ?

In [ ]:
#@title Repérer les entités nommées de chaque extrait (connexion Internet)
import spacy
nlp = spacy.load("fr_core_news_sm")                # le petit modèle français de spaCy
lignes = []                                         # une ligne par entité trouvée
for titre, texte in corpus.items():
    doc = nlp(" ".join(texte.split()))              # l'extrait sur une seule ligne
    for ent in doc.ents:                            # ent.text : l'entité ; ent.label_ : son type
        lignes.append({"extrait": titre, "entité": ent.text, "type": ent.label_})
entites = pd.DataFrame(lignes)
entites

**Ce que vous devez voir :** un tableau de 16 entités (résultats obtenus avec `fr_core_news_sm` 3.8.0 ; une autre version du modèle peut donner des résultats un peu différents).
- Hugo (paix) : les dix villes, plus l'Angleterre, l'Allemagne et la France, toutes en `LOC` (13 lieux), et « Assemblée législative » en `ORG`. C'est juste.
- Deux erreurs, en `MISC` : « Remarquez-le bien » (Hugo, misère) et « Un jour viendra » (Hugo, paix). La majuscule en début de phrase a trompé le modèle.
- Un oubli : « l'Europe » n'est pas repérée. Jaurès : aucune entité.

**Question :** pourquoi un modèle qui s'appuie sur les majuscules réussit-il si bien avec des noms de villes, et si mal avec « Un jour viendra » ?

Le mini-projet demande un **tableau des entités** par discours. La cellule suivante compte les entités par type (en lignes) et par extrait (en colonnes). Les extraits sans aucune entité sont gardés, avec des zéros.

**Avant d'exécuter :** quelle case du tableau aura le plus grand nombre ?

In [ ]:
#@title Compter les entités par type et par extrait (connexion Internet)
tableau_entites = pd.crosstab(entites["type"], entites["extrait"])
tableau_entites = tableau_entites.reindex(columns=list(corpus), fill_value=0)   # garder tous les extraits
tableau_entites

**Ce que vous devez voir :**

| type | Hugo 1849, la misère | Jaurès 1903, le courage | Hugo 1849, la paix |
|---|---|---|---|
| LOC | 0 | 0 | 13 |
| MISC | 1 | 0 | 1 |
| ORG | 0 | 0 | 1 |

La case la plus remplie est `LOC` pour Hugo (paix) : 13 lieux.

**Question :** peut-on conclure de ce tableau que Jaurès ne parle d'aucun lieu ni d'aucune personne ? Que faudrait-il vérifier, sur un extrait plus long ?

Pour finir, testez spaCy sur une phrase de votre choix. La phrase proposée est **inventée** pour l'exemple ; c'est celle de la leçon du mini-projet.

**Avant d'exécuter :** quelles entités, et de quels types, spaCy devrait-il trouver dans cette phrase ?

In [ ]:
#@title À MODIFIER : tester spaCy sur une phrase (connexion Internet)
phrase = "À Constantine, le ministre Karim Benali a évoqué avec l'UNESCO la rénovation de la médina." #@param {type:"string"}
for ent in nlp(phrase).ents:
    print(ent.text, "->", ent.label_)

**Ce que vous devez voir :**
```
Constantine -> LOC
Karim Benali -> PER
UNESCO -> ORG
```
**Question :** ajoutez une date à la fin de la phrase (« le 12 mars 2024 »), puis remplacez Constantine par Médéa et Karim Benali par un autre nom, masculin puis féminin. spaCy repère-t-il toujours tout ? Qu'en concluez-vous pour des discours prononcés en Algérie ?

## À vous

Modifiez seulement les champs, puis relancez les cellules à partir de celle que vous avez changée (ou tout le carnet : « Exécution → Tout exécuter »).
1. **Étape 9, une famille de mots :** tapez `*possible` dans `mot_cherche`. La concordance réunit « possible » et « impossible » : qui emploie l'un, qui emploie l'autre, et à propos de quoi ? Essayez aussi `jour`, `vie`, `peuple*`.
2. **Étape 6, les mots vides :** ajoutez `peut, dis, messieurs` dans `mots_vides_en_plus`, puis relancez les étapes 6 à 8. Qu'est-ce qui change dans le graphique et dans le tableau TF-IDF de Hugo ? Ces mots étaient-ils vraiment « vides » pour étudier sa rhétorique ?
3. **Étape 3, un quatrième extrait :** collez dans `mon_extrait` un court passage (une centaine de mots) d'un discours du domaine public que vous étudiez, ou un paragraphe écrit par vous sur la misère ou la paix, puis relancez tout le carnet. Quels mots de Hugo perdent leur statut de mots caractéristiques ?

## Pour le mini-projet : votre propre corpus

Pour votre mini-projet, remplacez nos trois extraits par **votre** corpus :
- préparez un fichier CSV (dans un tableur : « Enregistrer sous… → CSV UTF-8 ») avec **une ligne par discours** et au moins deux colonnes, `titre` et `texte` ; ajoutez les métadonnées demandées (`orateur`, `date`, `lieu`, `langue`, original ou traduction) ;
- nettoyez les textes : retirez ce qui n'est pas du discours (« applaudissements », en-têtes, numéros de page) ;
- cochez la case ci-dessous et exécutez la cellule : dans Colab, une fenêtre vous demande le fichier ; dans Jupyter, placez le fichier à côté du carnet sous le nom `mes_discours.csv`.

**Avant d'exécuter :** si vous laissez la case décochée, quel corpus la cellule va-t-elle utiliser ?

In [ ]:
#@title À MODIFIER : charger votre propre corpus (fichier CSV, colonnes titre et texte)
charger_mon_fichier = False #@param {type:"boolean"}
mes_discours = pd.DataFrame({"titre": list(corpus), "texte": list(corpus.values())})  # par défaut
if charger_mon_fichier:
    try:
        from google.colab import files       # n'existe que dans Colab
        envoi = files.upload()               # fenêtre pour choisir le fichier
        nom_fichier = list(envoi.keys())[0]
    except ImportError:                      # Jupyter : fichier placé à côté du carnet
        nom_fichier = "mes_discours.csv"
    mes_discours = pd.read_csv(nom_fichier)
print("Corpus utilisé :", len(mes_discours), "discours")

**Ce que vous devez voir :** avec la case décochée, `Corpus utilisé : 3 discours` (nos extraits). Avec votre fichier, le nombre de lignes de votre tableau.

La cellule suivante résume chaque discours du corpus chargé : sa taille, ses mots pleins les plus fréquents et ses mots les plus caractéristiques selon TF-IDF.

**Avant d'exécuter :** retrouvez-vous, pour nos extraits, des mots déjà vus aux étapes 6 et 8 ?

In [ ]:
#@title Résumer chaque discours : taille, mots fréquents, mots caractéristiques
vect = TfidfVectorizer(token_pattern=r"\w+", stop_words=sorted(mots_vides))
scores_corpus = pd.DataFrame(vect.fit_transform(mes_discours["texte"]).toarray(),
                             columns=vect.get_feature_names_out())
for i in range(len(mes_discours)):
    titre, texte = mes_discours["titre"][i], mes_discours["texte"][i]
    pleins = [m for m in mots_de(texte) if m not in mots_vides]
    caract = scores_corpus.loc[i].sort_values(ascending=False, kind="stable").head(5)
    print(f"{titre} : {len(mots_de(texte))} mots, {len(set(mots_de(texte)))} formes")
    print("    les plus fréquents :", [mot for mot, n in Counter(pleins).most_common(5)])
    print("    caractéristiques   :", list(caract.index))

**Ce que vous devez voir :**
```
Hugo 1849, la misère : 116 mots, 67 formes
    les plus fréquents : ['misère', 'peut', 'détruire', 'messieurs', 'souffrance']
    caractéristiques   : ['misère', 'détruire', 'peut', 'corps', 'dis']
Jaurès 1903, le courage : 152 mots, 80 formes
    les plus fréquents : ['courage', 'vie', 'comprendre', 'récompense', 'réserve']
    caractéristiques   : ['courage', 'vie', 'comprendre', 'récompense', 'réserve']
Hugo 1849, la paix : 131 mots, 78 formes
    les plus fréquents : ['jour', 'viendra', 'absurde', 'impossible', 'ouvrant']
    caractéristiques   : ['jour', 'viendra', 'absurde', 'impossible', 'ouvrant']
```
Avec nos trois extraits, les deux listes sont presque identiques ; avec un vrai corpus de plusieurs dizaines de discours, elles le seront beaucoup moins.

**Question :** pour votre mini-projet, laquelle des deux listes utiliserez-vous pour décrire le sujet d'un discours, et laquelle pour le distinguer des autres discours de votre corpus ?

## À rendre

Répondez par écrit, en français, en quelques lignes pour chaque question.
1. **Mots caractéristiques.** Choisissez un extrait. Quels sont ses trois mots les plus caractéristiques selon TF-IDF ? À l'aide de la concordance (étape 9), montrez comment l'orateur les emploie : anaphore, champ lexical, opposition…
2. **Compter ou comparer.** Comparez, pour cet extrait, le classement par fréquence (étape 6) et le classement TF-IDF (étape 8). Pourquoi se ressemblent-ils autant ici ? Dans quel corpus TF-IDF apporterait-il davantage ?
3. **Juger l'outil.** Relevez deux erreurs ou deux oublis de spaCy à l'étape 10 (si vous n'avez pas pu l'exécuter, appuyez-vous sur les résultats attendus). Que vous apprennent-ils sur la confiance à accorder à un outil automatique dans votre mini-projet, et comment le vérifieriez-vous ?

## Ce qu'il faut retenir

- Compter les **mots** (occurrences) et les **formes** suppose de définir ce qu'est un mot (espaces, apostrophes, traits d'union) ; les fréquences brutes sont dominées par les **mots vides**, et la liste de mots vides est un choix d'analyse.
- **TF-IDF** compare les textes : il met en avant les mots fréquents dans un texte et rares dans les autres. Il est d'autant plus utile que le corpus est grand ; la **concordance** (KWIC) ramène toujours du chiffre au texte.
- La **reconnaissance d'entités nommées** (spaCy) repère lieux, personnes et organisations par apprentissage automatique, mais elle se trompe : on la vérifie toujours sur un échantillon.

**Leçons du site :** [Mini-projet, sujet 3 : discours politiques](https://progremer04.github.io/canva-au-for-ang-/#mp-sujet-3) · [Chapitre 2 : lire et traiter des fichiers textes (fréquences, concordance)](https://progremer04.github.io/canva-au-for-ang-/#c2-fichiers)